# 🤖 NewsBot Intelligence System
## ITAI 2373 - Mid-Term Group Project

**Team Members:** Abraham BArreto
**Date:** [6/25/2026]

---

## 🎯 Project Overview

This notebook builds a complete NLP system that:

- 📰 **Processes** news articles with advanced text cleaning
- 🏷️ **Classifies** articles into six categories (Politics, Sports, Technology, Business, Entertainment, Health)
- 🔍 **Extracts** named entities (people, organizations, locations, dates, money)
- 😊 **Analyzes** sentiment and emotional tone
- 📊 **Generates** insights for business intelligence

### 📚 Module Integration Checklist
- [x] **Module 1:** NLP applications and real-world context
- [x] **Module 2:** Text preprocessing pipeline
- [x] **Module 3:** TF-IDF feature extraction
- [x] **Module 4:** POS tagging analysis
- [x] **Module 5:** Syntax parsing and semantic analysis
- [x] **Module 6:** Sentiment and emotion analysis
- [x] **Module 7:** Text classification system
- [x] **Module 8:** Named Entity Recognition

---

## 📦 Setup and Installation

Run the cell below **once** to install everything. In VS Code, select a Python kernel (top-right) before running.

In [ ]:
# Install required packages (run this cell first!)
%pip install -q spacy scikit-learn nltk pandas numpy matplotlib seaborn wordcloud
!python -m spacy download en_core_web_sm

# Download NLTK data
import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4',
            'vader_lexicon', 'averaged_perceptron_tagger',
            'averaged_perceptron_tagger_eng']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Could not download {pkg}: {e}")

print("✅ All packages installed successfully!")

In [ ]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter, defaultdict
import re
import warnings
warnings.filterwarnings('ignore')

# NLP Libraries
import spacy
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.tag import pos_tag

# Scikit-learn for machine learning
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import MaxAbsScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline

# Load spaCy model
nlp = spacy.load('en_core_web_sm')

# Plotting style
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

print("📚 All libraries imported successfully!")
print(f"🔧 spaCy model loaded: {nlp.meta['name']} v{nlp.meta['version']}")

## 📊 Data Loading and Exploration

### 🎯 Module 1: Understanding Our NLP Application

Before the technical work, here is the real-world context of the NewsBot Intelligence System. It addresses several business needs:

1. **Media Monitoring:** Automatically categorize and track news coverage
2. **Business Intelligence:** Extract key entities and sentiment trends
3. **Content Management:** Organize large volumes of news content
4. **Market Research:** Understand public sentiment about topics and entities

**Other real-world applications:** customer-support ticket routing, legal-document classification, social-media brand monitoring, financial-news trading signals, and academic-literature organization.

We use a real **BBC News dataset** from Kaggle (e.g. the `learn-ai-bbc` competition's `BBC News Train.csv`, ~1,490 articles across 5 categories: business, entertainment, politics, sport, tech). The loader below auto-detects whichever BBC CSV you downloaded and normalizes it to a standard schema. It also works with the synthetic `news_dataset.csv` if you prefer that.

In [ ]:
import os, glob
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================================
# SMART DATASET LOADER  —  auto-detects common Kaggle BBC news datasets
# ----------------------------------------------------------------------------
# Works with whichever of these you downloaded (just drop the CSV next to this
# notebook, or in your Downloads folder):
#   • learn-ai-bbc            -> "BBC News Train.csv"   (cols: ArticleId, Text, Category)
#   • yufengdev fulltext      -> "bbc-text.csv"         (cols: category, text)
#   • hgultekin bbcnewsarchive-> "bbc-news-data.csv"    (tab-sep: title, content, category)
#   • the synthetic dataset   -> "news_dataset.csv"     (already in final schema)
# It normalizes everything to: article_id, title, content, category, date, source
# ============================================================================

# 1) Find a dataset file --------------------------------------------------------
KNOWN_NAMES = [
    "news_dataset.csv", "BBC News Train.csv", "BBC_News_Train.csv",
    "bbc-text.csv", "bbc_text.csv", "bbc-news-data.csv", "bbc-news.csv",
]
search_dirs = [Path.cwd(), Path.home() / "Downloads", Path.home()]
found = None
for d in search_dirs:
    for name in KNOWN_NAMES:
        p = d / name
        if p.exists():
            found = p; break
    if found: break
# Fallback: any csv containing "bbc" in the current tree
if found is None:
    hits = [Path(p) for p in glob.glob(str(Path.cwd() / "**" / "*bbc*.csv"), recursive=True)]
    found = hits[0] if hits else None
if found is None:
    raise FileNotFoundError(
        "No dataset found. Download 'BBC News Train.csv' from "
        "https://www.kaggle.com/competitions/learn-ai-bbc/data and place it in "
        "this notebook's folder (or your Downloads folder)."
    )

# 2) Read it (auto-detect tab vs comma) -----------------------------------------
sep = "\t" if found.suffix == ".csv" and "\t" in found.read_text(encoding="utf-8", errors="ignore")[:2000] else ","
raw = pd.read_csv(found, sep=sep)
raw.columns = [c.strip() for c in raw.columns]
lower = {c.lower(): c for c in raw.columns}

def pick(*options):
    for o in options:
        if o in lower: return lower[o]
    return None

text_col  = pick("content", "text", "body", "article", "news")
cat_col   = pick("category", "categories", "label", "class", "topic")
title_col = pick("title", "headline", "heading")
id_col    = pick("article_id", "articleid", "id")
date_col  = pick("date", "published", "publish_date")
src_col   = pick("source", "publication", "publisher")

if text_col is None or cat_col is None:
    raise ValueError(f"Could not identify text/category columns in {list(raw.columns)}")

# 3) Normalize to the standard schema -------------------------------------------
df = pd.DataFrame()
df["content"]  = raw[text_col].astype(str).str.strip()
df["category"] = raw[cat_col].astype(str).str.strip().str.title()   # 'tech' -> 'Tech'
df["title"]    = (raw[title_col].astype(str).str.strip() if title_col
                  else df["content"].str.split().str[:12].str.join(" ") + "...")  # synth title
df["article_id"] = (raw[id_col] if id_col else range(1, len(df) + 1))
df["date"]   = raw[date_col] if date_col else "2005-01-01"   # BBC corpus is 2004-2005
df["source"] = raw[src_col] if src_col else "BBC News"

# 4) Clean: drop empties/dupes, enforce "substantial text" (>= 25 words) ---------
df = df[df["content"].str.split().str.len() >= 25]
df = df.drop_duplicates(subset="content").reset_index(drop=True)
df = df[["article_id", "title", "content", "category", "date", "source"]]

# 5) Report ----------------------------------------------------------------------
print("📊 Dataset loaded successfully!")
print(f"📂 Source file: {found.name}  (sep={'TAB' if sep==chr(9) else 'comma'})")
print(f"📈 Shape: {df.shape}")
print(f"🏷️ Categories ({df['category'].nunique()}): {sorted(df['category'].unique())}")
print(f"📝 Avg words/article: {df['content'].str.split().str.len().mean():.0f}")

# Requirement check
ok_count = len(df) >= 500
ok_cats  = df['category'].nunique() >= 4
print("\nRequirement check:")
print(f"   [{'PASS' if ok_count else 'FAIL'}] >= 500 articles: {len(df)}")
print(f"   [{'PASS' if ok_cats else 'FAIL'}] >= 4 categories: {df['category'].nunique()}")
df.head()


In [ ]:
# Basic dataset exploration
print("📊 DATASET OVERVIEW")
print("=" * 50)
print(f"Total articles: {len(df)}")
print(f"Unique categories: {df['category'].nunique()}")
print(f"Categories: {df['category'].unique().tolist()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Unique sources: {df['source'].nunique()}")

print("\n📈 CATEGORY DISTRIBUTION")
print("=" * 50)
category_counts = df['category'].value_counts()
print(category_counts)

# Visualize category distribution
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='category', order=category_counts.index)
plt.title('Distribution of News Categories')
plt.xlabel('Category'); plt.ylabel('Number of Articles')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### 💡 Student Task: Deeper Exploratory Data Analysis
We check for missing values, analyze text-length distribution, examine the source distribution, and look for data-quality issues.

In [ ]:
# 1. Check for missing values
print("🔍 MISSING VALUES CHECK")
print("=" * 50)
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"Duplicate titles: {df['title'].duplicated().sum()}")

# 2. Text length features
df['char_count'] = df['content'].str.len()
df['word_count'] = df['content'].str.split().str.len()
df['title_word_count'] = df['title'].str.split().str.len()

print("\n📏 TEXT LENGTH STATISTICS (content)")
print("=" * 50)
print(df[['char_count', 'word_count', 'title_word_count']].describe().round(1))

# 3. Visualize length distributions and source counts
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.histplot(df['word_count'], bins=20, kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Distribution of Article Word Count')
axes[0, 0].set_xlabel('Words per article')

sns.boxplot(data=df, x='category', y='word_count', ax=axes[0, 1])
axes[0, 1].set_title('Word Count by Category')
axes[0, 1].tick_params(axis='x', rotation=45)

source_counts = df['source'].value_counts().head(15)
source_counts.plot(kind='barh', ax=axes[1, 0])
axes[1, 0].set_title('Top 15 Sources by Article Count')
axes[1, 0].invert_yaxis()

avg_len = df.groupby('category')['word_count'].mean().sort_values()
avg_len.plot(kind='bar', ax=axes[1, 1], color='teal')
axes[1, 1].set_title('Average Word Count by Category')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].set_ylabel('Avg words')

plt.tight_layout()
plt.show()

print("\n💡 OBSERVATIONS:")
print(f"- No missing values; dataset is clean and balanced (20 per category).")
print(f"- Longest articles on average: {avg_len.idxmax()} ({avg_len.max():.0f} words).")
print(f"- Shortest articles on average: {avg_len.idxmin()} ({avg_len.min():.0f} words).")

## 🧹 Text Preprocessing Pipeline

### 🎯 Module 2: Advanced Text Preprocessing

We implement a preprocessing pipeline that cleans and normalizes our news articles.

**Key Steps:** Text cleaning (remove HTML, URLs, special characters) → Tokenization → Normalization (lowercase) → Stop-word removal → Lemmatization.

**Why it matters:** Without preprocessing, the model treats "Apple", "apple!", and "apple." as different tokens, inflating the vocabulary with noise and hurting downstream accuracy.

In [ ]:
# Initialize preprocessing tools
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """Clean raw text: lowercase, strip HTML/URLs/emails/special chars/extra whitespace."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', '', text)                              # HTML tags
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)  # URLs
    text = re.sub(r'\S+@\S+', '', text)                             # emails
    text = re.sub(r'[^a-zA-Z\s]', '', text)                         # keep letters + spaces
    text = re.sub(r'\s+', ' ', text).strip()                        # extra whitespace
    return text

def preprocess_text(text, remove_stopwords=True, lemmatize=True):
    """Full pipeline: clean -> tokenize -> remove stopwords -> lemmatize -> filter short tokens."""
    text = clean_text(text)
    if not text:
        return ""
    tokens = word_tokenize(text)
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    if lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    tokens = [t for t in tokens if len(t) > 2]
    return ' '.join(tokens)

# Test
sample_text = "Apple Inc. announced record quarterly earnings today! Visit https://apple.com for more info. #TechNews"
print("Original text:\n", sample_text)
print("\nCleaned text:\n", clean_text(sample_text))
print("\nFully preprocessed text:\n", preprocess_text(sample_text))

In [ ]:
# Apply preprocessing to the dataset
print("🧹 Preprocessing all articles...")

df['title_clean'] = df['title'].apply(clean_text)
df['content_clean'] = df['content'].apply(clean_text)
df['title_processed'] = df['title'].apply(preprocess_text)
df['content_processed'] = df['content'].apply(preprocess_text)

# Combine title + content for full-article analysis
df['full_text'] = df['title'] + ' ' + df['content']
df['full_text_processed'] = df['full_text'].apply(preprocess_text)

print("✅ Preprocessing complete!")

print("\n📝 BEFORE AND AFTER EXAMPLES")
print("=" * 60)
for i in range(min(3, len(df))):
    print(f"\nExample {i+1}:")
    print(f"Original:  {df.iloc[i]['full_text'][:100]}...")
    print(f"Processed: {df.iloc[i]['full_text_processed'][:100]}...")

#### 💡 Student Task: Analyze the Preprocessing Results
We compare text length before/after, count unique words, and find the most common words after preprocessing.

In [ ]:
# Length before vs after
df['words_before'] = df['full_text'].str.split().str.len()
df['words_after'] = df['full_text_processed'].str.split().str.len()

avg_before = df['words_before'].mean()
avg_after = df['words_after'].mean()
reduction = (1 - avg_after / avg_before) * 100

print("📊 PREPROCESSING IMPACT")
print("=" * 50)
print(f"Average words BEFORE preprocessing: {avg_before:.1f}")
print(f"Average words AFTER preprocessing:  {avg_after:.1f}")
print(f"Reduction: {reduction:.1f}%")

# Unique vocabulary before vs after
vocab_before = set(' '.join(df['full_text'].str.lower()).split())
vocab_after = set(' '.join(df['full_text_processed']).split())
print(f"\nUnique words BEFORE: {len(vocab_before)}")
print(f"Unique words AFTER:  {len(vocab_after)}")

# Most common words after preprocessing
all_words = ' '.join(df['full_text_processed']).split()
common = Counter(all_words).most_common(20)

print("\n🔥 TOP 20 WORDS AFTER PREPROCESSING")
print("=" * 50)
for word, cnt in common:
    print(f"  {word}: {cnt}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(['Before', 'After'], [avg_before, avg_after], color=['salmon', 'seagreen'])
axes[0].set_title('Average Words per Article: Before vs After')
axes[0].set_ylabel('Words')

words, counts = zip(*common)
axes[1].barh(words[::-1], counts[::-1], color='slateblue')
axes[1].set_title('Top 20 Most Common Words (after preprocessing)')
plt.tight_layout()
plt.show()

## 📊 Feature Extraction and Statistical Analysis

### 🎯 Module 3: TF-IDF Analysis

We extract numerical features using **TF-IDF** (Term Frequency–Inverse Document Frequency).

- **TF:** how often a word appears in a document
- **IDF:** how rare a word is across all documents
- **TF-IDF = TF × IDF:** balances frequency with uniqueness

**Business value:** TF-IDF surfaces the most distinctive terms for each news category.

In [ ]:
# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,   # cap vocabulary
    ngram_range=(1, 2),  # unigrams + bigrams
    min_df=2,            # ignore terms in < 2 docs
    max_df=0.8           # ignore terms in > 80% of docs
)

print("🔢 Creating TF-IDF features...")
tfidf_matrix = tfidf_vectorizer.fit_transform(df['full_text_processed'])
feature_names = tfidf_vectorizer.get_feature_names_out()

print("✅ TF-IDF matrix created!")
print(f"📊 Shape: {tfidf_matrix.shape}")
print(f"📝 Vocabulary size: {len(feature_names)}")
sparsity = (1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100
print(f"🔢 Sparsity: {sparsity:.2f}%")

# DataFrame for analysis
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
tfidf_df['category'] = df['category'].values

print("\n🔍 Sample TF-IDF features:")
print(tfidf_df.iloc[:3, :8])

In [ ]:
# Top TF-IDF terms per category
def get_top_tfidf_terms(category, n_terms=10):
    """Mean TF-IDF score per term for one category; returns top N."""
    category_data = tfidf_df[tfidf_df['category'] == category]
    mean_scores = category_data.drop('category', axis=1).mean().sort_values(ascending=False)
    return mean_scores.head(n_terms)

print("🏷️ TOP TF-IDF TERMS BY CATEGORY")
print("=" * 50)
categories = sorted(df['category'].unique())
category_terms = {}
for category in categories:
    top_terms = get_top_tfidf_terms(category, n_terms=10)
    category_terms[category] = top_terms
    print(f"\n📰 {category.upper()}:")
    for term, score in top_terms.items():
        print(f"  {term}: {score:.4f}")

#### 💡 Student Task: Visualize TF-IDF (word clouds, bar charts, heatmap)

In [ ]:
# Word clouds for each category
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, category in enumerate(categories):
    text_blob = ' '.join(df[df['category'] == category]['full_text_processed'])
    wc = WordCloud(width=500, height=300, background_color='white',
                   colormap='viridis', max_words=50).generate(text_blob)
    axes[i].imshow(wc, interpolation='bilinear')
    axes[i].set_title(f'{category}', fontsize=14, fontweight='bold')
    axes[i].axis('off')
plt.suptitle('Word Clouds by News Category', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Bar charts of top terms per category
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, category in enumerate(categories):
    top = category_terms[category]
    axes[i].barh(list(top.index)[::-1], list(top.values)[::-1], color=plt.cm.tab10(i))
    axes[i].set_title(f'Top TF-IDF Terms: {category}', fontweight='bold')
    axes[i].set_xlabel('Mean TF-IDF score')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: top distinctive terms across categories
top_terms_set = []
for category in categories:
    top_terms_set.extend(get_top_tfidf_terms(category, n_terms=6).index.tolist())
top_terms_set = list(dict.fromkeys(top_terms_set))  # unique, keep order

heat_data = tfidf_df.groupby('category')[top_terms_set].mean()
plt.figure(figsize=(16, 6))
sns.heatmap(heat_data, cmap='YlOrRd', annot=False, cbar_kws={'label': 'Mean TF-IDF'})
plt.title('Term Importance Across Categories (top distinctive terms)')
plt.xlabel('Term'); plt.ylabel('Category')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

print("💡 Insight: each category has clearly distinct high-TF-IDF terms,")
print("which is exactly what makes automatic classification feasible.")

## 🏷️ Part-of-Speech Analysis

### 🎯 Module 4: Grammatical Pattern Analysis

We analyze grammatical patterns across categories using **POS tagging**.

**Applications:** writing-style detection, content-quality assessment, and feature engineering for classification.

**Hypothesis:** Sports articles use more action verbs; business/politics articles use more proper nouns and numbers.

In [ ]:
def analyze_pos_patterns(text):
    """Tag tokens and return POS-tag proportions for the text."""
    if not text or pd.isna(text):
        return {}
    tokens = word_tokenize(str(text))
    pos_tags = pos_tag(tokens)
    pos_counts = Counter(tag for _, tag in pos_tags)
    total = len(pos_tags)
    if total == 0:
        return {}
    return {pos: count / total for pos, count in pos_counts.items()}

print("🏷️ Analyzing POS patterns...")
pos_results = []
for _, row in df.iterrows():
    pa = analyze_pos_patterns(row['full_text'])
    pa['category'] = row['category']
    pa['article_id'] = row['article_id']
    pos_results.append(pa)

pos_df = pd.DataFrame(pos_results).fillna(0)
print(f"✅ POS analysis complete! Found {len(pos_df.columns)-2} different POS tags")
print("\n📝 Sample POS analysis:")
print(pos_df.head())

In [ ]:
# POS patterns by category
print("📊 POS PATTERNS BY CATEGORY")
print("=" * 50)
pos_by_category = pos_df.groupby('category').mean(numeric_only=True)

major_pos = ['NN','NNS','NNP','NNPS','VB','VBD','VBG','VBN','VBP','VBZ',
             'JJ','JJR','JJS','RB','RBR','RBS','CD']
available_pos = [p for p in major_pos if p in pos_by_category.columns]
pos_summary = pos_by_category[available_pos]

print("\n🎯 Key POS patterns by category (proportions):")
print(pos_summary.round(4))

plt.figure(figsize=(12, 8))
sns.heatmap(pos_summary.T, annot=True, cmap='YlOrRd', fmt='.3f')
plt.title('POS Tag Proportions by News Category')
plt.xlabel('Category'); plt.ylabel('POS Tag')
plt.tight_layout()
plt.show()

#### 💡 Student Task: Answer the POS Analysis Questions
We answer the four analysis questions directly from the data.

In [ ]:
# Aggregate noun / verb / adjective / number groups
def group_sum(cols):
    cols = [c for c in cols if c in pos_by_category.columns]
    return pos_by_category[cols].sum(axis=1)

proper_nouns = group_sum(['NNP', 'NNPS'])
all_nouns    = group_sum(['NN', 'NNS', 'NNP', 'NNPS'])
action_verbs = group_sum(['VB', 'VBD', 'VBG', 'VBP', 'VBZ', 'VBN'])
adjectives   = group_sum(['JJ', 'JJR', 'JJS'])
numbers      = group_sum(['CD'])

summary = pd.DataFrame({
    'Proper Nouns': proper_nouns,
    'All Nouns': all_nouns,
    'Action Verbs': action_verbs,
    'Adjectives': adjectives,
    'Numbers (CD)': numbers,
}).round(4)

print("📋 GRAMMATICAL PROFILE BY CATEGORY")
print("=" * 60)
print(summary)

print("\n💡 ANSWERS TO ANALYSIS QUESTIONS:")
print(f"1. Highest proportion of proper nouns: {proper_nouns.idxmax()} ({proper_nouns.max():.3f})")
print(f"2. Most action verbs: {action_verbs.idxmax()} ({action_verbs.max():.3f})")
print(f"3. Most adjectives: {adjectives.idxmax()} ({adjectives.max():.3f})")
print(f"4. Most numbers (CD): {numbers.idxmax()} ({numbers.max():.3f})")

# Grouped bar chart
summary.plot(kind='bar', figsize=(14, 6))
plt.title('Grammatical Profile by News Category')
plt.ylabel('Proportion of tokens')
plt.xticks(rotation=45)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 🌳 Syntax Parsing and Semantic Analysis

### 🎯 Module 5: Understanding Sentence Structure

We use spaCy for **dependency parsing** to extract relationships between words — not just what words appear, but how they relate.

**Applications:** relationship extraction, event detection (who did what to whom), and information extraction.

In [ ]:
def extract_syntactic_features(text):
    """Extract dependency relations, noun phrases, subjects, and objects with spaCy."""
    if not text or pd.isna(text):
        return {}
    doc = nlp(str(text))
    features = {
        'num_sentences': len(list(doc.sents)),
        'num_tokens': len(doc),
        'dependency_relations': [],
        'noun_phrases': [],
        'subjects': [],
        'objects': []
    }
    for token in doc:
        if not token.is_space and not token.is_punct:
            features['dependency_relations'].append(token.dep_)
    for chunk in doc.noun_chunks:
        features['noun_phrases'].append(chunk.text.lower())
    for token in doc:
        if token.dep_ in ['nsubj', 'nsubjpass']:
            features['subjects'].append(token.text.lower())
        elif token.dep_ in ['dobj', 'iobj', 'pobj']:
            features['objects'].append(token.text.lower())
    features['dependency_counts'] = dict(Counter(features['dependency_relations']))
    return features

print("🌳 Performing syntactic analysis (first 5 articles for demo)...")
syntactic_results = []
for _, row in df.head(5).iterrows():
    feats = extract_syntactic_features(row['full_text'])
    feats['category'] = row['category']
    feats['article_id'] = row['article_id']
    syntactic_results.append(feats)

print("✅ Syntactic analysis complete!\n")
for i, r in enumerate(syntactic_results):
    print(f"📰 Article {i+1} ({r['category']}):")
    print(f"  Sentences: {r['num_sentences']} | Tokens: {r['num_tokens']}")
    print(f"  Noun phrases: {r['noun_phrases'][:3]}")
    print(f"  Subjects: {r['subjects'][:3]} | Objects: {r['objects'][:3]}\n")

In [ ]:
# Visualize dependency parse for one sample sentence
from spacy import displacy

sample_sentence = df.iloc[0]['content'].split('.')[0]  # first sentence
print(f"📝 Sample sentence: {sample_sentence}\n")
doc = nlp(sample_sentence)

# In VS Code/Jupyter this renders an SVG tree.
try:
    displacy.render(doc, style="dep", jupyter=True, options={'distance': 110})
except Exception:
    print("🔗 Dependency Relations:")
    for token in doc:
        if not token.is_space and not token.is_punct:
            print(f"  {token.text} --{token.dep_}--> {token.head.text}")

#### 💡 Student Task: Compare Syntactic Complexity Across Categories
We process all articles and compare sentence length, noun-phrase density, and most common dependency relations per category.

In [ ]:
# Full syntactic pass across the corpus (lightweight metrics)
print("🌳 Computing syntactic complexity for all articles...")
syn_rows = []
dep_by_cat = defaultdict(Counter)

for _, row in df.iterrows():
    doc = nlp(str(row['full_text']))
    n_sents = len(list(doc.sents)) or 1
    n_tokens = sum(1 for t in doc if not t.is_punct and not t.is_space)
    n_noun_chunks = len(list(doc.noun_chunks))
    for t in doc:
        if not t.is_punct and not t.is_space:
            dep_by_cat[row['category']][t.dep_] += 1
    syn_rows.append({
        'category': row['category'],
        'avg_sentence_length': n_tokens / n_sents,
        'noun_phrase_density': n_noun_chunks / n_tokens if n_tokens else 0,
        'num_sentences': n_sents,
    })

syn_df = pd.DataFrame(syn_rows)
complexity = syn_df.groupby('category')[['avg_sentence_length', 'noun_phrase_density', 'num_sentences']].mean().round(3)
print("\n📊 SYNTACTIC COMPLEXITY BY CATEGORY")
print("=" * 60)
print(complexity)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
complexity['avg_sentence_length'].sort_values().plot(kind='barh', ax=axes[0], color='darkcyan')
axes[0].set_title('Average Sentence Length (tokens) by Category')
complexity['noun_phrase_density'].sort_values().plot(kind='barh', ax=axes[1], color='indianred')
axes[1].set_title('Noun-Phrase Density by Category')
plt.tight_layout()
plt.show()

print("\n🔗 TOP 5 DEPENDENCY RELATIONS PER CATEGORY")
print("=" * 60)
for cat in sorted(dep_by_cat):
    top = dep_by_cat[cat].most_common(5)
    print(f"{cat}: {', '.join(f'{d}({c})' for d, c in top)}")

## 😊 Sentiment and Emotion Analysis

### 🎯 Module 6: Understanding Emotional Tone

We analyze sentiment using **VADER**, which returns `compound` (-1 to 1), plus `pos`, `neu`, and `neg` scores.

**Hypothesis:** different categories carry different emotional profiles — sports more positive, politics/health more negative.

In [ ]:
sia = SentimentIntensityAnalyzer()

def analyze_sentiment(text):
    """VADER sentiment with a human-readable label."""
    if not text or pd.isna(text):
        return {'compound': 0, 'pos': 0, 'neu': 1, 'neg': 0, 'sentiment_label': 'neutral'}
    scores = sia.polarity_scores(str(text))
    if scores['compound'] >= 0.05:
        scores['sentiment_label'] = 'positive'
    elif scores['compound'] <= -0.05:
        scores['sentiment_label'] = 'negative'
    else:
        scores['sentiment_label'] = 'neutral'
    return scores

print("😊 Analyzing sentiment...")
sentiment_results = []
for _, row in df.iterrows():
    ts = analyze_sentiment(row['title'])
    cs = analyze_sentiment(row['content'])
    fs = analyze_sentiment(row['full_text'])
    sentiment_results.append({
        'article_id': row['article_id'], 'category': row['category'],
        'title_sentiment': ts['compound'], 'title_label': ts['sentiment_label'],
        'content_sentiment': cs['compound'], 'content_label': cs['sentiment_label'],
        'full_sentiment': fs['compound'], 'full_label': fs['sentiment_label'],
        'pos_score': fs['pos'], 'neu_score': fs['neu'], 'neg_score': fs['neg'],
    })

sentiment_df = pd.DataFrame(sentiment_results)
print(f"✅ Sentiment analysis complete! Analyzed {len(sentiment_df)} articles\n")
print(sentiment_df[['category', 'full_sentiment', 'full_label']].head())

In [ ]:
# Sentiment patterns by category
print("📊 SENTIMENT ANALYSIS BY CATEGORY")
print("=" * 50)
sentiment_by_category = sentiment_df.groupby('category').agg(
    mean_sentiment=('full_sentiment', 'mean'),
    std_sentiment=('full_sentiment', 'std'),
    min_sentiment=('full_sentiment', 'min'),
    max_sentiment=('full_sentiment', 'max'),
    pos=('pos_score', 'mean'),
    neg=('neg_score', 'mean'),
).round(4)
print(sentiment_by_category)

sentiment_dist = sentiment_df.groupby(['category', 'full_label']).size().unstack(fill_value=0)
sentiment_dist_pct = (sentiment_dist.div(sentiment_dist.sum(axis=1), axis=0) * 100).round(1)
print("\n📊 Sentiment distribution (%) by category:")
print(sentiment_dist_pct)

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
sns.boxplot(data=sentiment_df, x='category', y='full_sentiment', ax=axes[0,0])
axes[0,0].set_title('Sentiment Score Distribution by Category')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].axhline(0, color='gray', ls='--', lw=1)

sentiment_dist_pct.plot(kind='bar', ax=axes[0,1], stacked=True)
axes[0,1].set_title('Sentiment Label Distribution by Category (%)')
axes[0,1].tick_params(axis='x', rotation=45)
axes[0,1].legend(title='Sentiment', bbox_to_anchor=(1.0, 1.0))

sentiment_df.groupby('category')[['pos_score','neg_score']].mean().plot(kind='bar', ax=axes[1,0])
axes[1,0].set_title('Average Positive vs Negative Scores by Category')
axes[1,0].tick_params(axis='x', rotation=45)
axes[1,0].legend(['Positive', 'Negative'])

pivot = sentiment_df.pivot_table(values='full_sentiment', index='category',
                                 columns='full_label', aggfunc='count', fill_value=0)
sns.heatmap(pivot, annot=True, fmt='d', ax=axes[1,1], cmap='YlGnBu')
axes[1,1].set_title('Sentiment Label Count Heatmap')
plt.tight_layout()
plt.show()

#### 💡 Student Task: Title vs Content Sentiment & Within-Category Variation

In [ ]:
# Title vs content sentiment
print("📊 TITLE vs CONTENT SENTIMENT")
print("=" * 50)
tc = sentiment_df.groupby('category')[['title_sentiment', 'content_sentiment']].mean().round(3)
tc['gap'] = (tc['title_sentiment'] - tc['content_sentiment']).round(3)
print(tc)

most_pos = sentiment_by_category['mean_sentiment'].idxmax()
most_neg = sentiment_by_category['mean_sentiment'].idxmin()
most_variable = sentiment_by_category['std_sentiment'].idxmax()

print(f"\n💡 FINDINGS:")
print(f"- Most positive category: {most_pos} ({sentiment_by_category['mean_sentiment'].max():.3f})")
print(f"- Most negative category: {most_neg} ({sentiment_by_category['mean_sentiment'].min():.3f})")
print(f"- Most variable (highest std): {most_variable} ({sentiment_by_category['std_sentiment'].max():.3f})")
print(f"- Titles are on average {'more positive' if tc['gap'].mean() > 0 else 'more negative'} than content.")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
tc[['title_sentiment','content_sentiment']].plot(kind='bar', ax=axes[0])
axes[0].set_title('Title vs Content Sentiment by Category')
axes[0].tick_params(axis='x', rotation=45)
axes[0].axhline(0, color='gray', ls='--', lw=1)

sns.violinplot(data=sentiment_df, x='category', y='full_sentiment', ax=axes[1])
axes[1].set_title('Within-Category Sentiment Spread')
axes[1].tick_params(axis='x', rotation=45)
axes[1].axhline(0, color='gray', ls='--', lw=1)
plt.tight_layout()
plt.show()

## 🏷️ Text Classification System

### 🎯 Module 7: Building the News Classifier

We build a multi-class classifier that automatically categorizes articles, comparing Naive Bayes, Logistic Regression, and SVM.

**Important design note:** TF-IDF values live on a 0–1 scale, but raw length features (character/word counts) are in the hundreds. If we combine them without scaling, the length features dominate and crush Logistic Regression and SVM. We therefore apply **MaxAbsScaler** to the combined feature matrix, which keeps all values in [0, 1] and stays non-negative so Naive Bayes still works.

In [ ]:
# Build the combined feature matrix
print("🔧 Preparing features for classification...")

X_tfidf = tfidf_matrix.toarray()                                    # TF-IDF
sentiment_features = sentiment_df[['full_sentiment', 'pos_score',   # sentiment
                                   'neu_score', 'neg_score']].values
length_features = np.column_stack([                                 # length
    df['full_text'].str.len(),
    df['full_text'].str.split().str.len(),
    df['title'].str.len(),
])

X_combined_raw = np.hstack([X_tfidf, sentiment_features, length_features])

# Make non-negative, then scale every column to [0,1] so no feature dominates.
X_combined_raw = np.abs(X_combined_raw)
scaler = MaxAbsScaler()
X_combined = scaler.fit_transform(X_combined_raw)

y = df['category'].values

print(f"✅ Feature matrix prepared! Shape: {X_combined.shape}")
print(f"🎯 Classes ({len(np.unique(y))}): {list(np.unique(y))}")

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\n📈 Training set: {X_train.shape[0]} | Test set: {X_test.shape[0]}")

In [ ]:
# Train and compare classifiers
print("🤖 Training multiple classifiers...")
classifiers = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=2000),
    'SVM': SVC(random_state=42, probability=True, kernel='linear'),
}

results, trained_models = {}, {}
for name, clf in classifiers.items():
    print(f"\n🔄 Training {name}...")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cv = cross_val_score(clf, X_train, y_train, cv=5, scoring='accuracy')
    results[name] = {'accuracy': acc, 'cv_mean': cv.mean(), 'cv_std': cv.std(),
                     'predictions': y_pred,
                     'probabilities': clf.predict_proba(X_test) if hasattr(clf,'predict_proba') else None}
    trained_models[name] = clf
    print(f"  ✅ Test Accuracy: {acc:.4f}")
    print(f"  📊 CV Score: {cv.mean():.4f} (+/- {cv.std()*2:.4f})")

print("\n🏆 CLASSIFIER COMPARISON")
print("=" * 50)
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Test Accuracy': [results[n]['accuracy'] for n in results],
    'CV Mean': [results[n]['cv_mean'] for n in results],
    'CV Std': [results[n]['cv_std'] for n in results],
}).round(4)
print(comparison_df.to_string(index=False))

best_model_name = comparison_df.loc[comparison_df['Test Accuracy'].idxmax(), 'Model']
print(f"\n🥇 Best performing model: {best_model_name}")

# Visualize comparison
comparison_df.set_index('Model')[['Test Accuracy','CV Mean']].plot(kind='bar', figsize=(10,5))
plt.title('Classifier Performance Comparison')
plt.ylabel('Accuracy'); plt.ylim(0,1); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# Detailed evaluation of the best model
best_model = trained_models[best_model_name]
best_predictions = results[best_model_name]['predictions']

print(f"📊 DETAILED EVALUATION: {best_model_name}")
print("=" * 60)
print("\n📋 Classification Report:")
print(classification_report(y_test, best_predictions))

cm = confusion_matrix(y_test, best_predictions, labels=np.unique(y))
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=np.unique(y), yticklabels=np.unique(y))
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout(); plt.show()

#### 💡 Student Task: Hyperparameter Tuning
We use `GridSearchCV` to tune the strongest model and report the improvement.

In [ ]:
# Hyperparameter tuning on Logistic Regression (robust on this feature set)
print("🔧 Tuning Logistic Regression with GridSearchCV...")
param_grid = {
    'C': [0.1, 0.5, 1.0, 2.0, 5.0],
    'solver': ['lbfgs', 'liblinear'],
}
grid = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=2000),
    param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
grid.fit(X_train, y_train)

tuned = grid.best_estimator_
tuned_acc = accuracy_score(y_test, tuned.predict(X_test))

print(f"\n✅ Best parameters: {grid.best_params_}")
print(f"📊 Best CV score: {grid.best_score_:.4f}")
print(f"🎯 Tuned test accuracy: {tuned_acc:.4f}")

baseline_lr = results['Logistic Regression']['accuracy']
print(f"\n📈 Baseline LR accuracy: {baseline_lr:.4f}")
print(f"📈 Tuned LR accuracy:    {tuned_acc:.4f}")
print(f"📈 Improvement:          {(tuned_acc - baseline_lr)*100:+.2f} percentage points")

# If tuned model beats the current best, adopt it for the rest of the pipeline.
if tuned_acc >= results[best_model_name]['accuracy']:
    best_model = tuned
    best_model_name = 'Logistic Regression (tuned)'
    results[best_model_name] = {'accuracy': tuned_acc,
                                'predictions': tuned.predict(X_test),
                                'probabilities': tuned.predict_proba(X_test)}
    trained_models[best_model_name] = tuned
    print(f"\n🥇 Adopted tuned model as best: {best_model_name}")

## 🔍 Named Entity Recognition

### 🎯 Module 8: Extracting Facts from News

We use spaCy NER to extract structured facts (people, organizations, locations, dates, money) from unstructured text.

**Business value:** enables queries like "all articles mentioning a company and its financials" or "track a public figure's mentions over time".

In [ ]:
def extract_entities(text):
    """Extract named entities with spaCy."""
    if not text or pd.isna(text):
        return []
    doc = nlp(str(text))
    return [{
        'text': ent.text, 'label': ent.label_,
        'start': ent.start_char, 'end': ent.end_char,
        'description': spacy.explain(ent.label_)
    } for ent in doc.ents]

print("🔍 Extracting named entities...")
all_entities, article_entities = [], []
for _, row in df.iterrows():
    entities = extract_entities(row['full_text'])
    article_entities.append({'article_id': row['article_id'], 'category': row['category'],
                             'entities': entities, 'entity_count': len(entities)})
    for e in entities:
        e = {**e, 'article_id': row['article_id'], 'category': row['category']}
        all_entities.append(e)

print(f"✅ Entity extraction complete! Total entities: {len(all_entities)}")
entities_df = pd.DataFrame(all_entities)
# Guard: ensure expected columns exist even if no entities were found
_ent_cols = ['text','label','start','end','description','article_id','category']
if entities_df.empty:
    entities_df = pd.DataFrame(columns=_ent_cols)
    print('⚠️ No entities found — check that the real en_core_web_sm model is loaded.')
print(f"🏷️ Entity types found: {sorted(entities_df['label'].unique())}")
print("\n📝 Sample entities:")
print(entities_df[['text', 'label', 'category']].head(10))

In [ ]:
# Entity analysis
print("📊 NAMED ENTITY ANALYSIS")
print("=" * 50)

if entities_df.empty:
    print("No entities were extracted. Make sure the real 'en_core_web_sm' model is")
    print("loaded (run the Setup cell). With the real model, BBC articles yield many entities.")
else:
    entity_counts = entities_df['label'].value_counts()
    print("\n🏷️ Entity type distribution:")
    print(entity_counts)

    entity_by_category = entities_df.groupby(['category', 'label']).size().unstack(fill_value=0)

    print("\n🔥 Most frequent entities:")
    freq = entities_df.groupby(['text', 'label']).size().sort_values(ascending=False).head(15)
    for (entity, label), count in freq.items():
        print(f"  {entity} ({label}): {count} mentions")

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    entity_counts.plot(kind='bar', ax=axes[0, 0], color='steelblue')
    axes[0, 0].set_title('Entity Type Distribution'); axes[0, 0].tick_params(axis='x', rotation=45)

    entities_df.groupby('category').size().plot(kind='bar', ax=axes[0, 1], color='coral')
    axes[0, 1].set_title('Total Entities per Category'); axes[0, 1].tick_params(axis='x', rotation=45)

    sns.heatmap(entity_by_category, annot=True, fmt='d', ax=axes[1, 0], cmap='YlOrRd')
    axes[1, 0].set_title('Entity Types by Category')

    entities_df['text'].value_counts().head(10).plot(kind='barh', ax=axes[1, 1], color='mediumseagreen')
    axes[1, 1].set_title('Most Mentioned Entities'); axes[1, 1].invert_yaxis()
    plt.tight_layout(); plt.show()


#### 💡 Student Task: Entity Co-occurrence Analysis
We find which entities are mentioned together in the same article — the basis of a knowledge graph.

In [ ]:
# Co-occurrence of PERSON/ORG/GPE entities within the same article
from itertools import combinations

cooccur = Counter()
for art in article_entities:
    ents = sorted(set(
        e['text'] for e in art['entities']
        if e['label'] in ('PERSON', 'ORG', 'GPE')
    ))
    for a, b in combinations(ents, 2):
        cooccur[(a, b)] += 1

print("🔗 TOP ENTITY CO-OCCURRENCES (PERSON / ORG / GPE)")
print("=" * 60)
top_pairs = cooccur.most_common(15)
if top_pairs:
    for (a, b), c in top_pairs:
        print(f"  {a}  <->  {b}: {c}")
else:
    print("  No repeated co-occurrences found.")

# Average entities-per-article by category (a richness measure)
if article_entities and any(a['entity_count'] for a in article_entities):
    richness = (pd.DataFrame(article_entities)
                .groupby('category')['entity_count'].mean().round(2).sort_values())
    print("\n📊 AVERAGE ENTITIES PER ARTICLE BY CATEGORY")
    print("=" * 60)
    print(richness)
    richness.plot(kind='barh', figsize=(9, 5), color='darkorange')
    plt.title('Average Named Entities per Article by Category')
    plt.xlabel('Avg entities'); plt.tight_layout(); plt.show()
else:
    print("\nNo entity richness to plot (no entities extracted).")


## 📈 Comprehensive Analysis and Insights

### 🎯 Bringing It All Together

We combine every analysis into a single insights report — where the business value of the pipeline emerges.

In [ ]:
def create_comprehensive_analysis():
    """Combine classification, sentiment, entity, and linguistic results into one report."""
    insights = {}

    insights['dataset_overview'] = {
        'total_articles': len(df),
        'categories': sorted(df['category'].unique().tolist()),
        'category_distribution': df['category'].value_counts().to_dict(),
        'avg_article_length': df['full_text'].str.len().mean(),
        'avg_words_per_article': df['full_text'].str.split().str.len().mean(),
    }

    insights['classification_performance'] = {
        'best_model': best_model_name,
        'best_accuracy': results[best_model_name]['accuracy'],
        'model_comparison': {n: results[n]['accuracy'] for n in results},
    }

    sent_by_cat = sentiment_df.groupby('category')['full_sentiment'].mean().to_dict()
    insights['sentiment_insights'] = {
        'most_positive_category': max(sent_by_cat, key=sent_by_cat.get),
        'most_negative_category': min(sent_by_cat, key=sent_by_cat.get),
        'sentiment_by_category': sent_by_cat,
        'overall_sentiment': sentiment_df['full_sentiment'].mean(),
    }

    insights['entity_insights'] = {
        'total_entities': len(entities_df),
        'unique_entities': entities_df['text'].nunique(),
        'entity_types': sorted(entities_df['label'].unique().tolist()),
        'entities_per_category': entities_df.groupby('category').size().to_dict(),
        'most_mentioned_entities': entities_df['text'].value_counts().head(5).to_dict(),
    }

    recs = []
    if insights['classification_performance']['best_accuracy'] > 0.8:
        recs.append("✅ High classification accuracy — ready for automated content routing.")
    else:
        recs.append("⚠️ Classification accuracy is moderate — more data or richer features would help.")
    pos = insights['sentiment_insights']['most_positive_category']
    neg = insights['sentiment_insights']['most_negative_category']
    recs.append(f"📊 {pos} articles skew most positive — good for uplifting content recommendations.")
    recs.append(f"📊 {neg} articles skew most negative — monitor for balanced coverage.")
    recs.append("🔍 Rich entity extraction enables advanced search and relationship analysis.")
    insights['business_recommendations'] = recs
    return insights

print("📊 Generating comprehensive analysis...")
analysis_results = create_comprehensive_analysis()
print("✅ Analysis complete!\n")
print("=" * 60)
print("📈 NEWSBOT INTELLIGENCE SYSTEM - COMPREHENSIVE REPORT")
print("=" * 60)

o = analysis_results['dataset_overview']
print(f"\n📊 DATASET OVERVIEW:")
print(f"  Total Articles: {o['total_articles']}")
print(f"  Categories: {', '.join(o['categories'])}")
print(f"  Avg Article Length: {o['avg_article_length']:.0f} characters")
print(f"  Avg Words per Article: {o['avg_words_per_article']:.0f} words")

p = analysis_results['classification_performance']
print(f"\n🤖 CLASSIFICATION PERFORMANCE:")
print(f"  Best Model: {p['best_model']}")
print(f"  Best Accuracy: {p['best_accuracy']:.4f}")

s = analysis_results['sentiment_insights']
print(f"\n😊 SENTIMENT INSIGHTS:")
print(f"  Most Positive Category: {s['most_positive_category']}")
print(f"  Most Negative Category: {s['most_negative_category']}")
print(f"  Overall Sentiment: {s['overall_sentiment']:.4f}")

e = analysis_results['entity_insights']
print(f"\n🔍 ENTITY INSIGHTS:")
print(f"  Total Entities: {e['total_entities']}")
print(f"  Unique Entities: {e['unique_entities']}")
print(f"  Entity Types: {', '.join(e['entity_types'])}")

print(f"\n💡 BUSINESS RECOMMENDATIONS:")
for i, rec in enumerate(analysis_results['business_recommendations'], 1):
    print(f"  {i}. {rec}")

## 🚀 Final System Integration

### 🎯 Building the Complete NewsBot Pipeline

We wrap everything in a single class that processes a brand-new article end to end: preprocess → classify → extract entities → analyze sentiment → generate insights.

**Note:** for new articles we recompute the same sentiment and length features used in training, then apply the **same scaler** so the feature space matches what the model learned.

In [ ]:
class NewsBotIntelligenceSystem:
    """Complete NewsBot Intelligence System for processing new articles."""

    def __init__(self, classifier, vectorizer, sentiment_analyzer, scaler):
        self.classifier = classifier
        self.vectorizer = vectorizer
        self.sentiment_analyzer = sentiment_analyzer
        self.scaler = scaler
        self.nlp = nlp

    def preprocess_article(self, title, content):
        full_text = f"{title} {content}"
        return full_text, preprocess_text(full_text)

    def _build_features(self, full_text, processed_text):
        """Recreate the exact training feature layout: TF-IDF + sentiment(4) + length(3)."""
        tfidf_vec = self.vectorizer.transform([processed_text]).toarray()
        s = analyze_sentiment(full_text)
        sentiment_vec = np.array([[s['compound'], s['pos'], s['neu'], s['neg']]])
        length_vec = np.array([[len(full_text), len(full_text.split()), len(full_text.split('  ')[0])]])
        combined = np.hstack([tfidf_vec, sentiment_vec, length_vec])
        combined = np.abs(combined)
        return self.scaler.transform(combined)

    def classify_article(self, full_text, processed_text):
        features = self._build_features(full_text, processed_text)
        prediction = self.classifier.predict(features)[0]
        probs = self.classifier.predict_proba(features)[0]
        class_probs = dict(zip(self.classifier.classes_, probs))
        return prediction, class_probs

    def extract_entities(self, text):
        return extract_entities(text)

    def analyze_sentiment(self, text):
        return analyze_sentiment(text)

    def generate_insights(self, category, entities, sentiment, category_probs):
        insights = []
        conf = max(category_probs.values())
        if conf > 0.8:
            insights.append(f"✅ High-confidence {category} classification ({conf:.1%})")
        else:
            insights.append(f"⚠️ Uncertain classification ({conf:.1%}) — consider manual review")
        c = sentiment['compound']
        if c > 0.1:
            insights.append(f"😊 Positive sentiment detected ({c:.3f})")
        elif c < -0.1:
            insights.append(f"😞 Negative sentiment detected ({c:.3f})")
        else:
            insights.append(f"😐 Neutral sentiment ({c:.3f})")
        if entities:
            types = {e['label'] for e in entities}
            insights.append(f"🔍 Found {len(entities)} entities across {len(types)} types")
            key = [e['text'] for e in entities if e['label'] in ('PERSON','ORG','GPE')][:3]
            if key:
                insights.append(f"🎯 Key entities: {', '.join(key)}")
        else:
            insights.append("ℹ️ No named entities detected")
        return insights

    def process_article(self, title, content):
        full_text, processed_text = self.preprocess_article(title, content)
        category, category_probs = self.classify_article(full_text, processed_text)
        entities = self.extract_entities(full_text)
        sentiment = self.analyze_sentiment(full_text)
        insights = self.generate_insights(category, entities, sentiment, category_probs)
        return {
            'title': title,
            'content': content[:200] + '...' if len(content) > 200 else content,
            'predicted_category': category,
            'category_confidence': max(category_probs.values()),
            'category_probabilities': category_probs,
            'entities': entities,
            'sentiment': sentiment,
            'insights': insights,
        }

newsbot = NewsBotIntelligenceSystem(
    classifier=best_model,
    vectorizer=tfidf_vectorizer,
    sentiment_analyzer=sia,
    scaler=scaler,
)
print("🤖 NewsBot Intelligence System initialized!")
print("✅ Ready to process new articles")

In [ ]:
# Test the complete system with new articles
print("🧪 TESTING NEWSBOT INTELLIGENCE SYSTEM")
print("=" * 60)

test_articles = [
    {'title': 'Microsoft Acquires AI Startup for $2 Billion',
     'content': "Microsoft Corporation announced today the acquisition of an artificial intelligence startup for $2 billion. CEO Satya Nadella said the deal will strengthen Microsoft's position in the AI market and enhance their cloud computing services."},
    {'title': 'Lakers Win Championship in Overtime Thriller',
     'content': 'The Los Angeles Lakers defeated the Boston Celtics 108-102 in overtime to win the NBA championship. LeBron James scored 35 points and was named Finals MVP for the fourth time in his career.'},
    {'title': 'New Climate Change Report Shows Alarming Trends',
     'content': 'Scientists at the United Nations released a comprehensive climate report showing accelerating global warming. The report warns that immediate action is needed to prevent catastrophic environmental changes.'},
    {'title': 'FDA Approves Breakthrough Cancer Treatment',
     'content': 'The Food and Drug Administration approved a new immunotherapy drug for advanced melanoma after clinical trials showed a 60 percent response rate. Doctors called the approval a major advance for patients.'},
]

for i, article in enumerate(test_articles, 1):
    print(f"\n📰 TEST ARTICLE {i}")
    print("-" * 40)
    result = newsbot.process_article(article['title'], article['content'])
    print(f"📰 Title: {result['title']}")
    print(f"🏷️ Predicted Category: {result['predicted_category']} ({result['category_confidence']:.1%} confidence)")
    print(f"\n📊 Category Probabilities:")
    for cat, prob in sorted(result['category_probabilities'].items(), key=lambda x: -x[1]):
        bar = '█' * int(prob * 20)
        print(f"  {cat:>14}: {prob:.3f} {bar}")
    print(f"\n😊 Sentiment: {result['sentiment']['sentiment_label']} (score: {result['sentiment']['compound']:.3f})")
    if result['entities']:
        print(f"\n🔍 Entities ({len(result['entities'])}):")
        for e in result['entities'][:5]:
            print(f"  {e['text']} ({e['label']}) - {e['description']}")
    print(f"\n💡 Insights:")
    for ins in result['insights']:
        print(f"  {ins}")

print("\n" + "=" * 60)
print("🎉 NewsBot Intelligence System testing complete!")

## 📝 Project Summary and Next Steps

### ✅ Module Integration Checklist
- [x] **Module 1:** Applied NLP to real-world news intelligence
- [x] **Module 2:** Implemented comprehensive text preprocessing
- [x] **Module 3:** Used TF-IDF for feature extraction and analysis
- [x] **Module 4:** Analyzed grammatical patterns with POS tagging
- [x] **Module 5:** Extracted syntactic relationships with dependency parsing
- [x] **Module 6:** Performed sentiment and emotion analysis
- [x] **Module 7:** Built and evaluated text classification models
- [x] **Module 8:** Implemented Named Entity Recognition

### 🚀 System Capabilities
The NewsBot can categorize articles, extract key entities, analyze sentiment, identify linguistic patterns, generate business insights, and process new articles end to end.

### 💼 Business Value
Useful for media companies (content routing), market research (sentiment + entity tracking), content management (smart organization), and business intelligence (trend analysis).

---

## 📋 Final Deliverables Checklist

### 📁 Code and Documentation
- [x] Complete Jupyter notebook with all analyses
- [x] Documented functions with docstrings
- [x] Clear markdown explanations for each section
- [ ] Organized GitHub repository structure
- [ ] README.md with project overview and setup instructions

### 📊 Analysis and Results
- [x] Dataset exploration, TF-IDF, POS, syntax, sentiment, classification, NER, integrated demo

### 📈 Visualizations
- [x] Category distribution, word clouds, POS heatmaps, sentiment plots, confusion matrix, entity charts

### 🎥 Presentation Materials
- [ ] 5–7 minute video demonstration
- [ ] Written report (3–4 pages)
- [ ] Individual reflection papers

---

## 🎓 Reflection Questions

For your individual reflection paper:

1. **Technical Mastery:** Which NLP techniques were most challenging? Most useful?
2. **Integration Challenges:** How did you handle combining multiple NLP tasks? (Hint: the feature-scaling issue in Module 7 is a great example.)
3. **Business Applications:** What real-world problems could this system solve?
4. **Ethical Considerations:** What are the risks of automated news analysis (bias, misclassification, surveillance)?
5. **Future Learning:** What NLP topics are you most excited to explore next?
6. **Team Collaboration:** How did you divide work and ensure quality?
7. **Portfolio Value:** How will you present this project to employers?

---

## 🔮 Future Enhancements
- Transformer models (BERT) for classification and NER
- Topic modeling (Module 9 preview)
- Trend/time-series analysis of entities and sentiment
- Streamlit dashboard or REST API deployment

---

## 🏆 Congratulations!
You've built a complete NLP pipeline integrating eight modules into one cohesive, production-style system.

**🚀 Ready for Module 9: Topic Modeling and Advanced Text Analysis!**